# Notebook 27 — AUS and Algorithmic vs Manual Lender Analysis

Tests whether the racial approval gap is smaller for algorithmically underwritten loans,
directly corroborating the discretion mechanism.

**Requires:** `data/processed/panel_with_dti.parquet` from NB25

**Runtime:** ~45 minutes

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

PROC = Path('../data/processed')
TABS = Path('../outputs/tables'); TABS.mkdir(exist_ok=True)
FIGS = Path('../outputs/figures'); FIGS.mkdir(exist_ok=True)
YEARS = [2020, 2021, 2022, 2023, 2024]
BLACK_CODE = 3
MAX_PER = 100  # per lender-race-year cap for DiD

print('='*65)
print('NB27: AUS AND ALGORITHMIC vs MANUAL LENDER ANALYSIS')
print('='*65)
print()
print('HMDA aus_1 field codes:')
print('  1 = Desktop Underwriter (DU) - Fannie Mae -- ALGORITHMIC')
print('  2 = Loan Prospector (LP) - Freddie Mac    -- ALGORITHMIC')
print('  3 = TOPS')
print('  4 = TOTAL Scorecard (FHA)')
print('  5 = GUS (USDA)')
print('  6 = Other automated')
print('  7 = Not applicable (manual underwriting)  -- MANUAL')
print('  1111 = Exempt')
print()
print('THEORY: If gap is smaller for algorithmic (DU/LP) than manual,')
print('human discretion amplifies differential treatment.')
print('If gap widened MORE after 2022 for manual lenders,')
print('tightening expanded human judgment and amplified the gap.')


NB27: AUS AND ALGORITHMIC vs MANUAL LENDER ANALYSIS

HMDA aus_1 field codes:
  1 = Desktop Underwriter (DU) - Fannie Mae -- ALGORITHMIC
  2 = Loan Prospector (LP) - Freddie Mac    -- ALGORITHMIC
  3 = TOPS
  4 = TOTAL Scorecard (FHA)
  5 = GUS (USDA)
  6 = Other automated
  7 = Not applicable (manual underwriting)  -- MANUAL
  1111 = Exempt

THEORY: If gap is smaller for algorithmic (DU/LP) than manual,
human discretion amplifies differential treatment.
If gap widened MORE after 2022 for manual lenders,
tightening expanded human judgment and amplified the gap.


In [3]:
# ---- AUS classification function ----
def classify_aus(val):
    if pd.isna(val):
        return 'Unknown'
    s = str(val).strip()
    if s in ['1', '2']:
        return 'GSE_Algorithmic'
    elif s == '7':
        return 'Manual'
    elif s in ['3', '4', '5', '6']:
        return 'Other_Automated'
    elif s in ['1111', 'Exempt', 'Exempted']:
        return 'Exempt'
    else:
        return 'Unknown'

# ---- Load panel_with_dti.csv saved by NB25 ----
csv_path = PROC / 'panel_with_dti.csv'

if not csv_path.exists():
    raise FileNotFoundError(
        'panel_with_dti.csv not found. Run NB25 Cell 2 first.'
    )

df_all = pd.read_csv(csv_path, dtype={'aus_1': str})
print('Loaded panel_with_dti.csv: {:,} rows'.format(len(df_all)))
print('Columns:', list(df_all.columns))

if 'aus_1' not in df_all.columns:
    raise ValueError('aus_1 column not found. Check NB25.')

df_all['aus_type'] = df_all['aus_1'].apply(classify_aus)

print()
print('AUS type distribution:')
print(df_all['aus_type'].value_counts().to_string())
print()
print('Approval rate by AUS type and race:')
pivot = df_all.groupby(['aus_type', 'black'])['approved'].agg(['mean', 'count'])
pivot['mean'] = (pivot['mean'] * 100).round(2)
print(pivot.to_string())

Loaded panel_with_dti.csv: 46,815,720 rows
Columns: ['lei', 'year', 'approved', 'black', 'income_n', 'loan_n', 'prop_n', 'ltv_n', 'dti_numeric', 'has_dti', 'aus_1']

AUS type distribution:
aus_type
GSE_Algorithmic    26192966
Other_Automated    18600407
Exempt              1161550
Manual               860797

Approval rate by AUS type and race:
                        mean     count
aus_type        black                 
Exempt          0      88.53   1109411
                1      71.14     52139
GSE_Algorithmic 0      91.62  23976662
                1      83.01   2216304
Manual          0      56.52    776763
                1      32.68     84034
Other_Automated 0      69.12  16174201
                1      54.09   2426206


In [4]:
# ----------------------------------------------------------------
# ANALYSIS 1: Within-lender gap by AUS type
# Compare mean within-lender Black-White gap across AUS categories
# ----------------------------------------------------------------
print()
print('='*65)
print('ANALYSIS 1: Within-lender gap by AUS type')
print('='*65)

aus_results = []
for aus_cat in ['GSE_Algorithmic', 'Manual', 'Other_Automated']:
    sub = df_all[(df_all['aus_type']==aus_cat) &
                 df_all['income_n'].notna() &
                 df_all['ltv_n'].notna()]
    if len(sub) < 500:
        print('  {}: insufficient obs ({:,}) -- skip'.format(aus_cat, len(sub)))
        continue

    lender_gaps = []
    for lei_val, grp in sub.groupby('lei'):
        b = grp[grp['black']==1]
        w = grp[grp['black']==0]
        if len(b) >= 10 and len(w) >= 10:
            gap = (w['approved'].mean() - b['approved'].mean()) * 100
            lender_gaps.append(gap)

    if not lender_gaps:
        continue

    m  = float(np.mean(lender_gaps))
    se = float(np.std(lender_gaps, ddof=1) / np.sqrt(len(lender_gaps)))
    aus_results.append({
        'AUS_Type': aus_cat,
        'N_lenders': len(lender_gaps),
        'Mean_gap_pp': round(m, 3),
        'SE': round(se, 3),
        'CI_L': round(m - 1.96*se, 3),
        'CI_U': round(m + 1.96*se, 3),
    })
    print('  {:20s}: gap={:+.2f}pp  SE={:.3f}  CI=[{:.2f},{:.2f}]  N_lenders={:,}'.format(
        aus_cat, m, se, m-1.96*se, m+1.96*se, len(lender_gaps)))

if aus_results:
    pd.DataFrame(aus_results).to_csv(TABS / 'table_27_aus_analysis.csv', index=False)
    print('Saved: table_27_aus_analysis.csv')
    print()
    print('INTERPRETATION:')
    print('  Smaller gap at GSE_Algorithmic than Manual -> discretion amplifies gap')
    print('  Consistent with Bartlett et al. (2022, JFE) FinTech finding')



ANALYSIS 1: Within-lender gap by AUS type
  GSE_Algorithmic     : gap=+6.91pp  SE=0.166  CI=[6.58,7.24]  N_lenders=2,085
  Manual              : gap=+13.40pp  SE=3.454  CI=[6.63,20.17]  N_lenders=15
  Other_Automated     : gap=+11.72pp  SE=0.240  CI=[11.25,12.19]  N_lenders=2,267
Saved: table_27_aus_analysis.csv

INTERPRETATION:
  Smaller gap at GSE_Algorithmic than Manual -> discretion amplifies gap
  Consistent with Bartlett et al. (2022, JFE) FinTech finding


In [5]:
# ----------------------------------------------------------------
# ANALYSIS 2: DiD by AUS type — does tightening amplify gap MORE
# for manually underwritten loans?
# ----------------------------------------------------------------
print()
print('='*65)
print('ANALYSIS 2: DiD (Black x Post2022) by AUS Type')
print('='*65)
print('If delta more negative for Manual than Algorithmic:')
print('-> Tightening amplified gap through human discretion channel')
print()

df_all['post2022']   = (df_all['year'] >= 2022).astype(int)
df_all['black_post'] = df_all['black'] * df_all['post2022']

triple_results = []

for aus_cat in ['GSE_Algorithmic', 'Manual', 'Other_Automated']:
    sub = df_all[(df_all['aus_type']==aus_cat) &
                 df_all['income_n'].notna() &
                 df_all['ltv_n'].notna()].copy()
    if len(sub) < 1000:
        print('  {}: insufficient obs -- skip'.format(aus_cat))
        continue

    # Cap per lender-year
    def cap_grp(g):
        b = g[g['black']==1]; w = g[g['black']==0]
        return pd.concat([
            b.sample(min(len(b), MAX_PER), random_state=42),
            w.sample(min(len(w), MAX_PER), random_state=42),
        ])

    df_cap = sub.groupby(['lei','year'], group_keys=False).apply(cap_grp)

    regs = ['black', 'post2022', 'black_post', 'income_n', 'ltv_n']
    df_cap = df_cap.dropna(subset=['approved', 'lei'] + regs)

    # FWL within-transformation
    lm = df_cap.groupby('lei')[['approved'] + regs].transform('mean')
    for c in ['approved'] + regs:
        df_cap[c+'_dm'] = df_cap[c] - lm[c]

    Xc = [c+'_dm' for c in regs]
    dr = df_cap[['approved_dm'] + Xc + ['lei']].dropna()
    X = dr[Xc].values
    y = dr['approved_dm'].values
    lei_arr = dr['lei'].values

    Xf = np.column_stack([np.ones(len(X)), X])
    coef, _, _, _ = np.linalg.lstsq(Xf, y, rcond=None)
    resid = y - Xf @ coef
    ul = np.unique(lei_arr); G = len(ul); n = len(y); k = Xf.shape[1]
    if G < 2: continue

    adj = (G/(G-1)) * ((n-1)/(n-k))
    try:    bread = np.linalg.inv(Xf.T @ Xf)
    except: bread = np.linalg.pinv(Xf.T @ Xf)
    meat = np.zeros((k, k))
    for lend in ul:
        idx = (lei_arr == lend)
        sc = Xf[idx].T @ resid[idx]
        meat += np.outer(sc, sc)
    vcov = adj * bread @ meat @ bread
    se_arr = np.sqrt(np.maximum(np.diag(vcov), 0))

    col_names = ['const'] + Xc
    bi = col_names.index('black_post_dm')
    delta = coef[bi] * 100
    dse   = se_arr[bi] * 100
    ts    = delta / dse if dse > 0 else 0
    pval  = 2 * (1 - stats.t.cdf(abs(ts), df=G-1))
    sig   = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else 'n.s.'

    triple_results.append({
        'AUS_Type': aus_cat, 'N_obs': n, 'N_lenders': G,
        'DiD_delta_pp': round(delta, 3), 'SE': round(dse, 3),
        'T_stat': round(ts, 3), 'P_value': round(pval, 4), 'Sig': sig,
    })
    print('  {:20s}: DiD delta={:+.3f}pp  SE={:.3f}  {}  N_lenders={:,}'.format(
        aus_cat, delta, dse, sig, G))

if triple_results:
    pd.DataFrame(triple_results).to_csv(TABS / 'table_27_fintech_did.csv', index=False)
    print()
    print('Saved: table_27_fintech_did.csv')

print()
print('NB27 COMPLETE')
print('Outputs:')
print('  outputs/tables/table_27_aus_analysis.csv')
print('  outputs/tables/table_27_fintech_did.csv')



ANALYSIS 2: DiD (Black x Post2022) by AUS Type
If delta more negative for Manual than Algorithmic:
-> Tightening amplified gap through human discretion channel

  GSE_Algorithmic     : DiD delta=-0.752pp  SE=0.193  ***  N_lenders=3,030
  Manual              : DiD delta=+1.151pp  SE=3.238  n.s.  N_lenders=79
  Other_Automated     : DiD delta=-0.000pp  SE=0.334  n.s.  N_lenders=3,590

Saved: table_27_fintech_did.csv

NB27 COMPLETE
Outputs:
  outputs/tables/table_27_aus_analysis.csv
  outputs/tables/table_27_fintech_did.csv
